In [6]:
import pandas as pd
df = pd.read_csv("data/subsidence_dataset.csv")
df["soil_missing"] = df["clay_pct"].isna().astype(int)
print(df["soil_missing"].sum(), "of", len(df), "cells flagged")   # should print 418
df.to_csv("data/subsidence_dataset_processed.csv", index=False)

418 of 5192 cells flagged


In [4]:
import os
import zipfile
import requests

BASE = "https://data.cnra.ca.gov/api/3/action"
HDRS = {"User-Agent": "Mozilla/5.0 (course project data download)"}
OUT = "data/insar_points.zip"
RESOURCE_URL = None   # paste a direct download link here to skip the automatic pick


def ckan(action, **params):
    r = requests.get(f"{BASE}/{action}", params=params, headers=HDRS, timeout=60)
    r.raise_for_status()
    return r.json()["result"]


# 1. License lines as DWR's portal reports them (for README and 6.3)
for pid in ["periodic-groundwater-level-measurements", "tre-altamira-insar-subsidence"]:
    try:
        p = ckan("package_show", id=pid)
        print(f"{pid}: license_title = {p.get('license_title')!r} | license_url = {p.get('license_url')!r}")
    except Exception as e:
        print(f"{pid}: could not read ({e})")

# 2. Find the InSAR dataset
try:
    pkg = ckan("package_show", id="tre-altamira-insar-subsidence")
except Exception:
    hits = ckan("package_search", q="TRE ALTAMIRA InSAR subsidence", rows=20)["results"]
    for h in hits:
        print("  candidate:", h["name"], "|", h["title"])
    pkg = next((h for h in hits if "altamira" in h["name"].lower()), None)
    if pkg is None:
        raise RuntimeError("Could not find the InSAR dataset. Download the file from the website instead.")
print("\nDataset:", pkg["name"], "|", pkg["title"])


# 3. List the files and pick the vertical-displacement point data
def score(r):
    text = f"{r.get('name', '')} {r.get('format', '')} {r.get('url', '')}".lower()
    s = 3 if "point" in text else 0
    s += 2 if "vertical" in text else 0
    s += 2 if ("zip" in text or "csv" in text) else 0
    s -= 3 if ("raster" in text or "tif" in text or "rate" in text) else 0
    s -= 2 if "location" in text else 0
    return s


res = sorted(pkg["resources"],
             key=lambda r: (score(r), r.get("last_modified") or r.get("created") or ""),
             reverse=True)
for r in res:
    print(f"  score {score(r):>2} | {r.get('name')} | {r.get('format')} | {r.get('size')} | {r.get('url')}")
url = RESOURCE_URL or res[0]["url"]

# 4. Download
print("\nDownloading:", url)
os.makedirs("data", exist_ok=True)
with requests.get(url, headers=HDRS, stream=True, timeout=(30, 300), allow_redirects=True) as r:
    r.raise_for_status()
    ctype = r.headers.get("Content-Type", "")
    total = int(r.headers.get("Content-Length", 0) or 0)
    print(f"Content-Type: {ctype}" + (f" | {total / 1e6:.0f} MB" if total else ""))
    if "html" in ctype.lower():
        raise RuntimeError("That link gives a web page, not a file (maybe a Box or landing page). "
                           "Open it in your browser and download the file by hand.")
    done = 0
    with open(OUT, "wb") as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
            done += len(chunk)
            print(f"  {done / 1e6:.0f} MB", end="\r")
print(f"\nSaved {OUT} ({os.path.getsize(OUT) / 1e6:.1f} MB)")
if zipfile.is_zipfile(OUT):
    with zipfile.ZipFile(OUT) as z:
        for i in z.infolist()[:10]:
            print("  in zip:", i.filename, f"{i.file_size / 1e6:.0f} MB")
else:
    print("Not a zip. If it is a plain CSV, rename it to data/insar_points.csv.")

periodic-groundwater-level-measurements: license_title = 'Other (Public Domain)' | license_url = None
tre-altamira-insar-subsidence: license_title = 'License not specified' | license_url = None

Dataset: tre-altamira-insar-subsidence | TRE ALTAMIRA InSAR Subsidence Data
  score  7 | Vertical Displacement Point Data | ZIP | 5262324358 | https://data.cnra.ca.gov/dataset/5e2d49e1-9ed0-425e-9f3e-2cda4a213c26/resource/6e855ae7-f365-4d40-984f-3c016b54f69d/download/verticaldisplacementpointdata.zip
  score  5 | Vertical Displacement Point Data Locations | ZIP | 265943387 | https://data.cnra.ca.gov/dataset/5e2d49e1-9ed0-425e-9f3e-2cda4a213c26/resource/8b5aec90-61bc-4ffa-abeb-c9f8261989ac/download/vertical_displacement_point_data_locations_2026q2.gdb.zip
  score  2 | Historical (2003-2015) ENVISAT and RADARSAT-2 Data | ZIP | 1428479039 | https://data.cnra.ca.gov/dataset/5e2d49e1-9ed0-425e-9f3e-2cda4a213c26/resource/69b51b6e-e0c7-4212-92bd-9fc5b052e32d/download/historical_2003-2015_envisat_radar

In [5]:
#!/usr/bin/env python3
"""
extract_insar.py -- pull the Tulare-Corcoran study-area points out of DWR's statewide
TRE ALTAMIRA "Vertical Displacement Point Data" download.

The download is a zip of many large CSVs (CALIFORNIA_DWR_xx_VERT.csv, 700+ MB each). This
reads every CSV straight from the zip (nothing is unpacked), keeps only the points inside
the study-area box taken from data/subsidence_dataset.csv, and writes them to one small CSV
that add_label.py can read.

For each file it first reads only the latitude/longitude columns to see whether the file
covers the study area at all, and only parses the full rows of files that do.

Usage (from the repo folder, Anaconda Prompt, env activated)
  python extract_insar.py
  python extract_insar.py --src data/insar_points.zip --grid data/subsidence_dataset.csv
  python extract_insar.py --src data/verticaldisplacementpointdata      # folder of CSVs extracted with 7-Zip

Then
  python add_label.py --insar data/insar_study_area.csv --inspect
"""
import argparse
import hashlib
import json
import re
import time
import warnings
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

LAT_NAMES = ["latitude", "lat", "point_y", "y"]
LON_NAMES = ["longitude", "lon", "long", "lng", "point_x", "x"]
READ = dict(low_memory=False, encoding_errors="replace")


def find_col(cols, names, kind, override=None):
    if override:
        if override not in cols:
            raise SystemExit(f"--{kind}-col {override!r} is not a column. Columns: {list(cols)[:25]}")
        return override
    lower = {str(c).lower(): c for c in cols}
    for n in names:
        if n in lower:
            return lower[n]
    raise SystemExit(f"Could not find a {kind} column. Columns start with {list(cols)[:25]}. "
                     f"Pass --{kind}-col NAME.")


def is_date_name(name):
    s = str(name).strip()
    digits = re.sub(r"\D", "", s)
    if len(digits) == 8:
        d = pd.to_datetime(digits, format="%Y%m%d", errors="coerce")
        return bool(pd.notna(d) and 2010 <= d.year <= 2035)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        d = pd.to_datetime(s.lstrip("Dd_"), errors="coerce")
    return bool(pd.notna(d) and 2010 <= d.year <= 2035 and not s.isdigit())


def date_value(name):
    s = str(name).strip()
    digits = re.sub(r"\D", "", s)
    if len(digits) == 8:
        return pd.to_datetime(digits, format="%Y%m%d", errors="coerce")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return pd.to_datetime(s.lstrip("Dd_"), errors="coerce")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()


class Source:
    """A zip of CSVs, a folder of CSVs, or one CSV."""

    def __init__(self, path):
        p = Path(path)
        if not p.exists():
            raise SystemExit(f"{path} does not exist.")
        self.zf = None
        if p.is_dir():
            self.members = sorted(p.rglob("*.csv"))
            self.names = [str(m.relative_to(p)) for m in self.members]
        elif p.suffix.lower() == ".zip":
            self.zf = zipfile.ZipFile(p)
            self.members = sorted((i for i in self.zf.infolist() if i.filename.lower().endswith(".csv")),
                                  key=lambda i: i.filename)
            self.names = [i.filename for i in self.members]
        else:
            self.members, self.names = [p], [p.name]
        if not self.members:
            raise SystemExit(f"No .csv files found in {path}.")

    def open(self, i):
        m = self.members[i]
        try:
            return self.zf.open(m) if self.zf else open(m, "rb")
        except NotImplementedError as e:
            raise SystemExit(
                f"Python cannot read this zip's compression method ({e}). Extract the zip with 7-Zip "
                f"(right-click > 7-Zip > Extract to ...) into a folder inside data\\, then rerun with "
                f"--src <that folder>.")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--src", default="data/insar_points.zip")
    ap.add_argument("--grid", default="data/subsidence_dataset.csv")
    ap.add_argument("--out", default="data/insar_study_area.csv")
    ap.add_argument("--log", default="data/insar_extract_log.json")
    ap.add_argument("--margin", type=float, default=0.01, help="degrees added around the grid box")
    ap.add_argument("--crs", default="EPSG:4326", help="CRS of the InSAR coordinate columns")
    ap.add_argument("--lat-col"); ap.add_argument("--lon-col")
    ap.add_argument("--start", default="2020-01-01"); ap.add_argument("--end", default="2024-12-31")
    ap.add_argument("--pad-days", type=int, default=180,
                    help="also keep date columns this many days either side of --start/--end (plus the first "
                         "column in the file, as a baseline check)")
    ap.add_argument("--all-dates", action="store_true", help="keep every date column (large output)")
    ap.add_argument("--chunk", type=int, default=100_000)
    ap.add_argument("--no-hash", action="store_true", help="skip the SHA-256 of the source file")
    a = ap.parse_args()

    grid = pd.read_csv(a.grid, usecols=["longitude", "latitude"])
    xmin, xmax = grid.longitude.min() - a.margin, grid.longitude.max() + a.margin
    ymin, ymax = grid.latitude.min() - a.margin, grid.latitude.max() + a.margin
    print(f"Study-area box: lon {xmin:.4f} to {xmax:.4f}, lat {ymin:.4f} to {ymax:.4f}")

    to_ll = None
    if a.crs.upper() != "EPSG:4326":
        from pyproj import Transformer
        to_ll = Transformer.from_crs(a.crs, "EPSG:4326", always_xy=True)

    def to_lonlat(df, lon_c, lat_c):
        lon = pd.to_numeric(df[lon_c], errors="coerce").to_numpy()
        lat = pd.to_numeric(df[lat_c], errors="coerce").to_numpy()
        return to_ll.transform(lon, lat) if to_ll is not None else (lon, lat)

    src = Source(a.src)
    print(f"{len(src.members)} CSV file(s) found in {a.src}\n")
    kept, log_files, t_all = [], [], time.time()
    out_cols = None

    for i, name in enumerate(src.names):
        t0 = time.time()
        with src.open(i) as fh:
            cols = list(pd.read_csv(fh, nrows=0, **READ).columns)
        lat_c = find_col(cols, LAT_NAMES, "lat", a.lat_col)
        lon_c = find_col(cols, LON_NAMES, "lon", a.lon_col)
        date_cols = sorted((date_value(c), c) for c in cols if is_date_name(c))
        n_dates = len(date_cols)
        if n_dates == 0:
            raise SystemExit(f"{name}: no date-like column names. Columns start with {cols[:25]}")
        if a.all_dates:
            keep_dates = [c for _, c in date_cols]
        else:
            lo_d = pd.Timestamp(a.start) - pd.Timedelta(days=a.pad_days)
            hi_d = pd.Timestamp(a.end) + pd.Timedelta(days=a.pad_days)
            keep_dates = [date_cols[0][1]] + [c for d, c in date_cols if lo_d <= d <= hi_d and c != date_cols[0][1]]
            if len(keep_dates) < 3:
                raise SystemExit(f"{name}: date columns run {date_cols[0][0].date()} to {date_cols[-1][0].date()}, "
                                 f"which does not reach the window {a.start} to {a.end}. Change --start/--end.")
        keep_cols = [c for c in cols if not is_date_name(c)] + keep_dates

        # pass 1: coordinates only -> which rows are in the study area, and what the file covers
        hit_rows, n_rows, span = [], 0, [np.inf, -np.inf, np.inf, -np.inf]
        with src.open(i) as fh:
            for chunk in pd.read_csv(fh, usecols=[lon_c, lat_c], chunksize=500_000, **READ):
                lon, lat = to_lonlat(chunk, lon_c, lat_c)
                if len(lat) and np.isfinite(lat).any():
                    span = [min(span[0], np.nanmin(lat)), max(span[1], np.nanmax(lat)),
                            min(span[2], np.nanmin(lon)), max(span[3], np.nanmax(lon))]
                m = (lon >= xmin) & (lon <= xmax) & (lat >= ymin) & (lat <= ymax)
                hit_rows.append(np.flatnonzero(m) + n_rows)
                n_rows += len(chunk)
        hit_rows = np.concatenate(hit_rows) if hit_rows else np.array([], dtype=int)
        msg = (f"[{i + 1}/{len(src.names)}] {name}: {n_rows:,} points, {len(hit_rows):,} in the study area "
               f"(covers lat {span[0]:.2f}..{span[1]:.2f}, lon {span[2]:.2f}..{span[3]:.2f}; "
               f"{n_dates} dates {date_cols[0][0].date()} to {date_cols[-1][0].date()})")

        # pass 2: full rows, only for files that reach the study area
        if len(hit_rows):
            got, base = [], 0
            with src.open(i) as fh:
                for chunk in pd.read_csv(fh, usecols=keep_cols, chunksize=a.chunk, **READ):
                    pos = np.arange(base, base + len(chunk))
                    sel = np.isin(pos, hit_rows)
                    if sel.any():
                        got.append(chunk.loc[sel].assign(source_file=name))
                    base += len(chunk)
            kept.extend(got)
            out_cols = out_cols or list(got[0].columns)
        print(f"{msg}; {time.time() - t0:.0f} s")
        log_files.append({"file": name, "points": int(n_rows), "in_study_area": int(len(hit_rows)),
                          "lat_range": [float(span[0]), float(span[1])], "lon_range": [float(span[2]), float(span[3])],
                          "date_columns_in_file": int(n_dates), "date_range": [str(date_cols[0][0].date()), str(date_cols[-1][0].date())],
                          "date_columns_kept": len(keep_dates)})

    if not kept:
        raise SystemExit("No points fell inside the study-area box in any file. Check --crs and the "
                         "lat/lon columns (the line above for each file shows what area it covers).")
    out = pd.concat(kept, ignore_index=True)
    lat_c, lon_c = find_col(out.columns, LAT_NAMES, "lat", a.lat_col), find_col(out.columns, LON_NAMES, "lon", a.lon_col)
    n_before = len(out)
    out = out.drop_duplicates(subset=[lat_c, lon_c]).reset_index(drop=True)
    print(f"\n{n_before:,} points kept; {n_before - len(out):,} repeated coordinates (same point in more than "
          f"one file) removed -> {len(out):,} points.")
    Path(a.out).parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(a.out, index=False)
    dates = [c for c in out.columns if is_date_name(c)]
    print(f"Saved {a.out} ({Path(a.out).stat().st_size / 1e6:.1f} MB): {len(out):,} points, {len(dates)} date "
          f"columns ({dates[0]} ... {dates[-1]}). Total time {(time.time() - t_all) / 60:.1f} min.")
    print("Non-date columns kept:", [c for c in out.columns if c not in dates])

    src_path = Path(a.src)
    log = {"run_at": datetime.now().isoformat(timespec="seconds"), "src": a.src,
           "src_bytes": src_path.stat().st_size if src_path.is_file() else None,
           "src_sha256": None, "bbox_lonlat": [xmin, ymin, xmax, ymax], "files": log_files,
           "points_kept": int(n_before), "points_after_dedup": int(len(out)), "output": a.out}
    if not a.no_hash and src_path.is_file():
        print("Hashing the source file for the lineage record (about a minute)...")
        log["src_sha256"] = sha256(a.src)
    Path(a.log).write_text(json.dumps(log, indent=2))
    print(f"Wrote {a.log}")


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--src SRC] [--grid GRID] [--out OUT] [--log LOG] [--margin MARGIN] [--crs CRS]
                             [--lat-col LAT_COL] [--lon-col LON_COL] [--start START] [--end END] [--pad-days PAD_DAYS]
                             [--all-dates] [--chunk CHUNK] [--no-hash]
ipykernel_launcher.py: error: unrecognized arguments: -f C:\Users\m.rahaman\AppData\Roaming\jupyter\runtime\kernel-447b40c5-a0c2-4dd5-af4b-cead52ed9bb8.json


SystemExit: 2

C:\Users\m.rahaman\.conda\envs\aml\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [9]:
"""Fetch NEON's own land-cover class (nlcdClass) and plot type for the 94 study plots.

Run on any computer with internet and Python 3 (no packages to install):
    python get_plot_landcover_local.py
Output, in the same folder:
    plot_landcover.csv       plotID, site, nlcdClass, plotType, plotSubtype
    plot_landcover_raw.json  every NEON location property per plot (for checking)
Then upload plot_landcover.csv (and plot_landcover_raw.json if nlcdClass is empty) to the chat.
Uses the public NEON locations API; no token needed.
"""
import csv, json, re, sys, time, urllib.request, urllib.error

PLOTS = [
    "BART_036", "BART_040", "BART_042", "BART_071", "BART_073", "BLAN_032", "BLAN_035", "BLAN_036",
    "BLAN_044", "BLAN_047", "DELA_037", "DELA_038", "DELA_040", "DELA_041", "DELA_042", "GRSM_010",
    "GRSM_018", "GRSM_055", "GRSM_057", "GRSM_058", "GRSM_059", "GRSM_060", "HARV_033", "HARV_034",
    "HARV_038", "HARV_039", "HARV_040", "JERC_047", "JERC_048", "JERC_049", "JERC_050", "JERC_051",
    "LENO_061", "LENO_062", "LENO_063", "LENO_064", "LENO_065", "LENO_066", "LENO_067", "LENO_068",
    "LENO_069", "LENO_070", "MLBS_061", "MLBS_062", "MLBS_063", "MLBS_064", "MLBS_065", "MLBS_066",
    "MLBS_067", "MLBS_068", "MLBS_069", "MLBS_070", "MLBS_071", "MLBS_072", "MLBS_073", "MLBS_074",
    "MLBS_075", "MLBS_076", "SERC_043", "SERC_044", "SERC_045", "SERC_047", "SERC_048", "SERC_049",
    "SERC_050", "SERC_051", "SERC_052", "SERC_053", "SERC_054", "SERC_055", "SERC_067", "SERC_073",
    "TALL_001", "TALL_002", "TALL_003", "TALL_005", "TALL_006", "TALL_007", "TALL_008", "TALL_009",
    "TALL_012", "TALL_013", "TALL_015", "TALL_016", "TALL_017", "TALL_018", "TALL_020", "TALL_022",
    "TALL_025", "TALL_042", "TALL_044", "TALL_047", "TALL_054", "TALL_055",
]

def get(pid):
    url = "https://data.neonscience.org/api/v0/locations/" + pid
    for attempt in range(4):
        try:
            with urllib.request.urlopen(url, timeout=30) as r:
                return json.loads(r.read().decode("utf-8"))["data"]
        except (urllib.error.URLError, TimeoutError, ValueError) as e:
            err = e
            time.sleep(2 * (attempt + 1))
    print("FAILED", pid, err)
    return None

def pick(props, pattern):
    for name, value in props.items():
        if re.search(pattern, name, re.I):
            return value
    return ""

rows, raw = [], {}
for k, pid in enumerate(PLOTS, 1):
    data = get(pid)
    props = {}
    if data:
        for p in data.get("locationProperties", []):
            props[p["locationPropertyName"]] = p["locationPropertyValue"]
    raw[pid] = props
    rows.append(dict(plotID=pid, site=pid.split("_")[0],
                     nlcdClass=pick(props, r"nlcd|land ?cover|vegetation type"),
                     plotType=pick(props, r"plot ?type"),
                     plotSubtype=pick(props, r"sub ?type")))
    if k == 1:
        print("Property names seen for", pid, ":", list(props))
    if k % 10 == 0:
        print(k, "of", len(PLOTS), "done")
    time.sleep(0.2)

with open("plot_landcover.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["plotID", "site", "nlcdClass", "plotType", "plotSubtype"])
    w.writeheader(); w.writerows(rows)
with open("plot_landcover_raw.json", "w", encoding="utf-8") as f:
    json.dump(raw, f, indent=1)

counts = {}
for r in rows:
    counts[(r["site"], r["nlcdClass"])] = counts.get((r["site"], r["nlcdClass"]), 0) + 1
for (s, c), n in sorted(counts.items()):
    print(f"{s:5s} {c or '(empty)':25s} {n}")
print("Wrote plot_landcover.csv and plot_landcover_raw.json")

FAILED BART_036 HTTP Error 400: Bad Request
Property names seen for BART_036 : []
FAILED BART_040 HTTP Error 400: Bad Request
FAILED BART_042 HTTP Error 400: Bad Request


KeyboardInterrupt: 